# 🛒 E-Commerce Sales Analysis — Data Science Capstone Project
### Indian E-Commerce Dataset (10,000 Orders | 2022–2024)

**Topics Covered:** Python Core · Intermediate Python · NumPy · Pandas · Matplotlib · Seaborn  
**Dataset:** `ecommerce_sales.csv` — 10,000 rows, 17 columns  
**Estimated Time:** 8–12 hours

---
## 📋 Project Overview
This capstone project simulates a real-world data science workflow using an Indian e-commerce sales dataset.  
You will clean, explore, analyse, and visualise data to extract actionable business insights.

### Learning Objectives
| Module | Skills |
|--------|--------|
| 1. Setup & Loading | File I/O, Python basics, pandas basics |
| 2. Data Inspection | dtypes, shape, head/tail, info |
| 3. Data Cleaning | Missing values, type casting, duplicates |
| 4. Python Core & Intermediate | List comprehensions, lambda, map/filter, OOP |
| 5. NumPy | Arrays, vectorised ops, stats |
| 6. Pandas EDA | GroupBy, pivot, merge, resample |
| 7. Matplotlib | Line, bar, pie, scatter, subplots |
| 8. Seaborn | Heatmap, boxplot, violin, pairplot, FacetGrid |
| 9. Business Insights | KPIs, cohort analysis, summary |
| 10. GitHub Push | Git workflow |


---
## 📦 Module 1 — Setup & Data Loading

In [ ]:
# Install any missing libraries (run once)
# !pip install pandas numpy matplotlib seaborn faker

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print("✅ All libraries imported successfully!")
print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")


In [ ]:
# ── Load the dataset ──────────────────────────────────────
df = pd.read_csv('ecommerce_sales.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


---
## 🔍 Module 2 — Data Inspection

In [ ]:
# Shape, dtypes, basic info
print("Shape :", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)


In [ ]:
# Statistical summary
df.describe(include='all').T


In [ ]:
# Missing value audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])


In [ ]:
# Unique values for categorical columns
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"{col:20s}: {df[col].nunique()} unique values → {df[col].dropna().unique()[:5]}")


---
## 🧹 Module 3 — Data Cleaning

In [ ]:
# ── 3.1 Fix data types ───────────────────────────────────
df['order_date'] = pd.to_datetime(df['order_date'])
df['customer_age'] = df['customer_age'].fillna(df['customer_age'].median())
df['customer_gender'] = df['customer_gender'].fillna('Unknown')
df['discount_pct'] = df['discount_pct'].fillna(0)

print("✅ Types fixed")
print(df[['order_date','customer_age','customer_gender','discount_pct']].dtypes)


In [ ]:
# ── 3.2 Add derived date columns ─────────────────────────
df['year']    = df['order_date'].dt.year
df['month']   = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%b')
df['quarter'] = df['order_date'].dt.quarter
df['day_of_week'] = df['order_date'].dt.day_name()
df['week'] = df['order_date'].dt.isocalendar().week.astype(int)

print("✅ Date columns added")
df[['order_date','year','month','quarter','day_of_week']].head(3)


In [ ]:
# ── 3.3 Duplicate check ──────────────────────────────────
dups = df.duplicated(subset='order_id').sum()
print(f"Duplicate order IDs: {dups}")
df = df.drop_duplicates(subset='order_id')
print(f"Dataset after dedup: {df.shape}")


In [ ]:
# ── 3.4 Validate numeric ranges ──────────────────────────
print("unit_price   :", df['unit_price'].min(), "–", df['unit_price'].max())
print("quantity     :", df['quantity'].min(), "–", df['quantity'].max())
print("discount_pct :", df['discount_pct'].min(), "–", df['discount_pct'].max())
print("total_price  :", df['total_price'].min(), "–", df['total_price'].max())
print("\nOrder status counts:")
print(df['order_status'].value_counts())


---
## 🐍 Module 4 — Python Core & Intermediate

### 4.1 List Comprehensions & Lambda Functions

In [ ]:
# List comprehension — classify age groups
df['age_group'] = ['Senior' if a >= 50 else 'Adult' if a >= 30 else 'Young Adult'
                   for a in df['customer_age']]

print(df['age_group'].value_counts())


In [ ]:
# Lambda + map — revenue tier
classify_revenue = lambda x: 'High' if x >= 10000 else ('Medium' if x >= 2000 else 'Low')
df['revenue_tier'] = df['total_price'].map(classify_revenue)
print(df['revenue_tier'].value_counts())


### 4.2 Filter & Reduce

In [ ]:
from functools import reduce

# filter() — high value delivered orders
high_value = list(filter(lambda x: x > 20000, df['total_price'].tolist()))
print(f"Orders above ₹20,000: {len(high_value):,}")

# reduce() — cumulative sum of top-5 cities revenue
top5_rev = (df.groupby('customer_city')['total_price'].sum()
              .nlargest(5).tolist())
cumulative = reduce(lambda a, b: a + b, top5_rev)
print(f"\nTop-5 cities cumulative revenue: ₹{cumulative:,.0f}")


### 4.3 OOP — Order Analytics Class

In [ ]:
class OrderAnalytics:
    """Encapsulates common order-level analytics."""

    def __init__(self, dataframe):
        self.df = dataframe.copy()

    def total_revenue(self, status='Delivered'):
        subset = self.df[self.df['order_status'] == status]
        return subset['total_price'].sum()

    def avg_order_value(self):
        return self.df['total_price'].mean()

    def top_categories(self, n=3):
        return (self.df.groupby('category')['total_price']
                       .sum().nlargest(n))

    def conversion_rate(self):
        """% of orders that were delivered."""
        delivered = (self.df['order_status'] == 'Delivered').sum()
        return round(delivered / len(self.df) * 100, 2)

    def summary(self):
        print("=" * 45)
        print("       E-Commerce Analytics Summary")
        print("=" * 45)
        print(f"  Total Orders       : {len(self.df):,}")
        print(f"  Total Revenue      : ₹{self.total_revenue():,.0f}")
        print(f"  Avg Order Value    : ₹{self.avg_order_value():,.2f}")
        print(f"  Delivery Rate      : {self.conversion_rate()}%")
        print("\n  Top 3 Categories by Revenue:")
        for cat, rev in self.top_categories().items():
            print(f"    {cat:<20} ₹{rev:>12,.0f}")
        print("=" * 45)

analytics = OrderAnalytics(df)
analytics.summary()


### 4.4 Exception Handling & File I/O

In [ ]:
import json

def export_summary(dataframe, filepath='summary_report.json'):
    try:
        summary = {
            'total_orders'  : int(len(dataframe)),
            'total_revenue' : float(dataframe['total_price'].sum().round(2)),
            'avg_order_val' : float(dataframe['total_price'].mean().round(2)),
            'date_range'    : {
                'from': str(dataframe['order_date'].min().date()),
                'to'  : str(dataframe['order_date'].max().date())
            },
            'top_category'  : dataframe.groupby('category')['total_price'].sum().idxmax()
        }
        with open(filepath, 'w') as f:
            json.dump(summary, f, indent=4)
        print(f"✅ Summary exported to {filepath}")
        return summary
    except Exception as e:
        print(f"❌ Export failed: {e}")

summary = export_summary(df)
print(json.dumps(summary, indent=4))


---
## 🔢 Module 5 — NumPy Operations

In [ ]:
# ── 5.1 Create NumPy arrays from DataFrame columns ───────
prices    = np.array(df['unit_price'])
quantities = np.array(df['quantity'])
discounts = np.array(df['discount_pct'])
totals    = np.array(df['total_price'])

print("Shape  :", prices.shape)
print("Dtype  :", prices.dtype)


In [ ]:
# ── 5.2 Vectorised calculations ─────────────────────────
# Revenue after discount (vectorised — no Python loop)
calculated_totals = (prices * quantities * (1 - discounts / 100)).round(2)

# Check accuracy (should be ~0 mean error)
diff = np.abs(calculated_totals - totals)
print(f"Max difference from stored total_price: ₹{diff.max():.4f}")
print(f"Mean difference: ₹{diff.mean():.4f}")


In [ ]:
# ── 5.3 Descriptive statistics with NumPy ────────────────
print("Revenue Statistics (NumPy)")
print("-" * 35)
stats = {
    'Mean'        : np.mean(totals),
    'Median'      : np.median(totals),
    'Std Dev'     : np.std(totals),
    'Variance'    : np.var(totals),
    'Min'         : np.min(totals),
    'Max'         : np.max(totals),
    '25th pct'    : np.percentile(totals, 25),
    '75th pct'    : np.percentile(totals, 75),
    'IQR'         : np.percentile(totals, 75) - np.percentile(totals, 25),
}
for k, v in stats.items():
    print(f"  {k:<14}: ₹{v:>12,.2f}")


In [ ]:
# ── 5.4 NumPy boolean indexing & masking ────────────────
high_value_mask  = totals > np.percentile(totals, 90)
print(f"Top 10% orders (> ₹{np.percentile(totals,90):,.0f}): {high_value_mask.sum():,}")
print(f"Revenue from top 10%: ₹{totals[high_value_mask].sum():,.0f}")
print(f"Revenue share: {totals[high_value_mask].sum()/totals.sum()*100:.1f}%")


In [ ]:
# ── 5.5 Correlation matrix with NumPy ────────────────────
numeric_cols = ['unit_price', 'quantity', 'discount_pct', 'total_price', 'customer_age']
numeric_matrix = np.column_stack([df[c] for c in numeric_cols])
corr_matrix = np.corrcoef(numeric_matrix.T)

print("Correlation Matrix (NumPy):")
print(pd.DataFrame(corr_matrix, index=numeric_cols, columns=numeric_cols).round(3))


---
## 🐼 Module 6 — Pandas EDA

### 6.1 GroupBy Aggregations

In [ ]:
# Revenue and order count by category
cat_stats = (df.groupby('category')
               .agg(
                   total_orders=('order_id','count'),
                   total_revenue=('total_price','sum'),
                   avg_order_value=('total_price','mean'),
                   avg_discount=('discount_pct','mean'),
                   avg_rating=('rating','mean')
               )
               .sort_values('total_revenue', ascending=False)
               .round(2))

cat_stats['revenue_share_%'] = (cat_stats['total_revenue'] /
                                 cat_stats['total_revenue'].sum() * 100).round(2)
cat_stats


### 6.2 Pivot Tables

In [ ]:
# Revenue by Category × Year
pivot_cat_year = df.pivot_table(
    values='total_price',
    index='category',
    columns='year',
    aggfunc='sum'
).round(0)

pivot_cat_year['Total'] = pivot_cat_year.sum(axis=1)
pivot_cat_year = pivot_cat_year.sort_values('Total', ascending=False)
pivot_cat_year.style.format('₹{:,.0f}').background_gradient(cmap='Blues', subset=[2022,2023,2024])


In [ ]:
# Payment method usage by city
pivot_pay_city = df.pivot_table(
    values='order_id',
    index='customer_city',
    columns='payment_method',
    aggfunc='count',
    fill_value=0
)
pivot_pay_city


### 6.3 Time Series Resampling

In [ ]:
# Monthly revenue trend
monthly = (df.set_index('order_date')['total_price']
             .resample('ME').sum()
             .reset_index())
monthly.columns = ['month', 'revenue']
monthly['rolling_3m'] = monthly['revenue'].rolling(3).mean()

print("Monthly Revenue (last 6 months):")
print(monthly.tail(6).to_string(index=False))


### 6.4 Multi-Level GroupBy & Transform

In [ ]:
# Customer RFM (Recency, Frequency, Monetary) table
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm = (df[df['order_status']=='Delivered']
       .groupby('customer_id')
       .agg(
           recency   = ('order_date', lambda x: (snapshot_date - x.max()).days),
           frequency = ('order_id', 'count'),
           monetary  = ('total_price', 'sum')
       )
       .reset_index())

# Score 1-5
for col in ['recency','frequency','monetary']:
    labels = [5,4,3,2,1] if col=='recency' else [1,2,3,4,5]
    rfm[col+'_score'] = pd.qcut(rfm[col], 5, labels=labels, duplicates='drop').astype(int)

rfm['rfm_score'] = (rfm['recency_score'].astype(str) +
                    rfm['frequency_score'].astype(str) +
                    rfm['monetary_score'].astype(str))

print("RFM Sample:")
print(rfm.head(5).to_string(index=False))
print(f"\nTotal customers scored: {len(rfm):,}")


### 6.5 String Operations

In [ ]:
# Extract product keyword from product_name using str methods
df['product_keyword'] = df['product_name'].str.upper().str.replace(' ', '_')

# Count products with '5' in name (e.g., Rice 5kg)
with_number = df['product_name'].str.contains(r'\d').sum()
print(f"Products with numbers in name: {with_number}")

# Most common starting letter of product names
df['product_name'].str[0].value_counts().head(5)


---
## 📊 Module 7 — Matplotlib Visualisations

In [ ]:
# ── 7.1 Monthly Revenue Line Chart ───────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(monthly['month'], monthly['revenue']/1e6,
        marker='o', linewidth=2, color='steelblue', label='Monthly Revenue')
ax.plot(monthly['month'], monthly['rolling_3m']/1e6,
        linewidth=2.5, linestyle='--', color='tomato', label='3-Month Rolling Avg')

ax.fill_between(monthly['month'], monthly['revenue']/1e6,
                alpha=0.15, color='steelblue')

ax.set_title('Monthly Revenue Trend (2022–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (₹ Millions)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:.1f}M'))
plt.tight_layout()
plt.savefig('plot_monthly_trend.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.2 Category Revenue Bar Chart ───────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

cat_rev = df.groupby('category')['total_price'].sum().sort_values()
colors  = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(cat_rev)))

bars = ax.barh(cat_rev.index, cat_rev.values / 1e6, color=colors, edgecolor='white')
ax.bar_label(bars, labels=[f'₹{v/1e6:.1f}M' for v in cat_rev.values], padding=4)

ax.set_title('Total Revenue by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue (₹ Millions)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('plot_category_revenue.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.3 Payment Method Pie Chart ─────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

pay_counts = df['payment_method'].value_counts()
explode = [0.05 if i==0 else 0 for i in range(len(pay_counts))]
wedges, texts, autotexts = ax.pie(
    pay_counts.values, labels=pay_counts.index,
    autopct='%1.1f%%', explode=explode,
    startangle=140, pctdistance=0.82
)
for at in autotexts:
    at.set_fontsize(9)

ax.set_title('Payment Method Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_payment_pie.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.4 Unit Price vs Total Price Scatter ────────────────
fig, ax = plt.subplots(figsize=(9, 5))

cats = df['category'].unique()
palette = plt.cm.tab10(np.linspace(0, 1, len(cats)))

for cat, color in zip(cats, palette):
    sub = df[df['category'] == cat]
    ax.scatter(sub['unit_price'], sub['total_price'],
               alpha=0.25, s=18, color=color, label=cat)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Unit Price vs Total Price (log scale)', fontsize=14, fontweight='bold')
ax.set_xlabel('Unit Price (₹)')
ax.set_ylabel('Total Price (₹)')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('plot_scatter.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.5 Subplots Dashboard ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('E-Commerce Dashboard', fontsize=16, fontweight='bold', y=1.01)

# Top-left: Order status donut
status_counts = df['order_status'].value_counts()
colors_s = ['#2ecc71','#e74c3c','#f39c12','#3498db']
axes[0,0].pie(status_counts.values, labels=status_counts.index,
              autopct='%1.1f%%', colors=colors_s,
              wedgeprops=dict(width=0.5))
axes[0,0].set_title('Order Status')

# Top-right: Orders by day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = df['day_of_week'].value_counts().reindex(dow_order)
axes[0,1].bar(range(7), dow_counts.values, color='#3498db', alpha=0.8)
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
axes[0,1].set_title('Orders by Day of Week')
axes[0,1].set_ylabel('Order Count')

# Bottom-left: Age distribution histogram
axes[1,0].hist(df['customer_age'].dropna(), bins=25, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[1,0].axvline(df['customer_age'].median(), color='red', linestyle='--', label=f"Median: {df['customer_age'].median():.0f}")
axes[1,0].set_title('Customer Age Distribution')
axes[1,0].set_xlabel('Age')
axes[1,0].legend()

# Bottom-right: Top 5 cities
top_cities = df.groupby('customer_city')['total_price'].sum().nlargest(5)
axes[1,1].barh(top_cities.index, top_cities.values/1e6, color='#1abc9c')
axes[1,1].set_title('Revenue by City (Top 5)')
axes[1,1].set_xlabel('Revenue (₹M)')

plt.tight_layout()
plt.savefig('plot_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 🎨 Module 8 — Seaborn Visualisations

In [ ]:
# ── 8.1 Correlation Heatmap ──────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

corr = df[['unit_price','quantity','discount_pct','total_price','customer_age','rating']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size':10})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 8.2 Boxplot — Revenue by Category ───────────────────
fig, ax = plt.subplots(figsize=(12, 6))

order_cat = (df.groupby('category')['total_price']
               .median().sort_values(ascending=False).index.tolist())

sns.boxplot(data=df, x='category', y='total_price',
            order=order_cat, palette='Set2', ax=ax,
            flierprops={'marker':'o','markersize':2,'alpha':0.3})
ax.set_yscale('log')
ax.set_title('Revenue Distribution by Category (log scale)', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Total Price (₹)')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig('plot_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 8.3 Violin Plot — Rating by Category ─────────────────
delivered = df[df['order_status']=='Delivered'].dropna(subset=['rating'])

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=delivered, x='category', y='rating',
               palette='pastel', inner='quartile', ax=ax)
ax.set_title('Rating Distribution by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Rating (1–5)')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig('plot_violin.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 8.4 Seaborn Heatmap — Revenue by Month × Category ────
pivot = df.pivot_table(values='total_price', index='month_name',
                       columns='category', aggfunc='sum')

month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
pivot = pivot.reindex(month_order)

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(pivot/1e3, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, ax=ax, annot_kws={'size':8},
            cbar_kws={'label':'Revenue (₹ Thousands)'})
ax.set_title('Monthly Revenue Heatmap by Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Month')
plt.tight_layout()
plt.savefig('plot_monthly_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 8.5 FacetGrid — Revenue by Gender across Categories ──
g = sns.FacetGrid(df[df['customer_gender'].isin(['Male','Female'])],
                  col='customer_gender', height=5, aspect=1.1)
g.map_dataframe(sns.histplot, x='total_price', bins=30,
                color='steelblue', log_scale=True)
g.set_axis_labels('Total Price (₹, log)', 'Count')
g.set_titles(col_template='{col_name} Customers')
g.figure.suptitle('Revenue Distribution by Gender', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_facetgrid.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 8.6 Pairplot — Key Numeric Features ──────────────────
sample = df[['unit_price','total_price','discount_pct','customer_age','rating']].dropna().sample(1000, random_state=42)
g = sns.pairplot(sample, diag_kind='kde', plot_kws={'alpha':0.3,'s':15})
g.figure.suptitle('Pairplot of Key Numeric Features (n=1,000 sample)', y=1.01, fontsize=13)
plt.savefig('plot_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()


---
## 💡 Module 9 — Business Insights & KPIs

In [ ]:
# ── 9.1 Key Performance Indicators ───────────────────────
delivered_df = df[df['order_status']=='Delivered']

kpis = {
    'Total Orders'          : len(df),
    'Delivered Orders'      : len(delivered_df),
    'Delivery Rate (%)'     : round(len(delivered_df)/len(df)*100, 2),
    'Total Revenue (₹)'     : round(delivered_df['total_price'].sum(), 2),
    'Avg Order Value (₹)'   : round(delivered_df['total_price'].mean(), 2),
    'Avg Rating'            : round(df['rating'].mean(), 2),
    'Avg Discount (%)'      : round(df['discount_pct'].mean(), 2),
    'Total Unique Customers': df['customer_id'].nunique(),
    'Return Rate (%)'       : round((df['order_status']=='Returned').sum()/len(df)*100, 2),
}

print("
📊 KEY PERFORMANCE INDICATORS")
print("=" * 45)
for k, v in kpis.items():
    print(f"  {k:<30}: {v:>12,}")


In [ ]:
# ── 9.2 Monthly Revenue Growth Rate ──────────────────────
monthly['mom_growth'] = monthly['revenue'].pct_change() * 100
monthly['yoy_label'] = pd.to_datetime(monthly['month']).dt.strftime('%Y-%m')

print("Month-over-Month Revenue Growth (last 12 months):")
print(monthly[['yoy_label','revenue','mom_growth']].tail(12)
      .rename(columns={'yoy_label':'Month','revenue':'Revenue (₹)','mom_growth':'MoM Growth (%)'})
      .to_string(index=False))


In [ ]:
# ── 9.3 Customer Segment Analysis ────────────────────────
seg_stats = (df.groupby('age_group')
               .agg(orders=('order_id','count'),
                    avg_spend=('total_price','mean'),
                    avg_discount=('discount_pct','mean'))
               .round(2))

print("\nCustomer Segment Analysis:")
print(seg_stats)


In [ ]:
# ── 9.4 Top Revenue Products ─────────────────────────────
top_products = (df.groupby(['category','product_name'])
                  .agg(orders=('order_id','count'),
                       revenue=('total_price','sum'))
                  .reset_index()
                  .sort_values('revenue', ascending=False)
                  .head(10))

top_products['revenue'] = top_products['revenue'].map('₹{:,.0f}'.format)
print("\nTop 10 Products by Revenue:")
print(top_products.to_string(index=False))


In [ ]:
# ── 9.5 Final Insights Chart — YoY Category Growth ───────
yoy = (df.groupby(['year','category'])['total_price']
         .sum().unstack().fillna(0))

growth = ((yoy.loc[2024] - yoy.loc[2022]) / yoy.loc[2022] * 100).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in growth.values]
bars = ax.barh(growth.index, growth.values, color=colors, edgecolor='white')
ax.bar_label(bars, labels=[f'{v:.1f}%' for v in growth.values], padding=4)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Revenue Growth: 2022 → 2024 by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Growth (%)')
plt.tight_layout()
plt.savefig('plot_yoy_growth.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 🚀 Module 10 — Pushing to GitHub

### Step-by-Step Git Workflow

```bash
# Step 1: Initialize Git in your project folder
cd ecommerce-capstone
git init

# Step 2: Create .gitignore
echo "__pycache__/" >> .gitignore
echo "*.pyc" >> .gitignore
echo ".ipynb_checkpoints/" >> .gitignore
echo "summary_report.json" >> .gitignore

# Step 3: Stage all files
git add ecommerce_sales.csv
git add capstone_notebook.ipynb
git add PROJECT_GUIDELINES.docx
git add README.md
git add .gitignore

# Step 4: Commit
git commit -m "Initial commit: E-Commerce Capstone Project"

# Step 5: Connect to GitHub (create repo on github.com first)
git remote add origin https://github.com/YOUR_USERNAME/ecommerce-capstone.git
git branch -M main

# Step 6: Push
git push -u origin main
```

### README.md Starter Template

```markdown
# 🛒 E-Commerce Sales Capstone Project

## About
A complete data science capstone project using Python, NumPy, Pandas,
Matplotlib and Seaborn on an Indian e-commerce dataset (10,000 orders).

## Files
| File | Description |
|------|-------------|
| `ecommerce_sales.csv` | Dataset — 10k rows, 17 columns |
| `capstone_notebook.ipynb` | Full analysis notebook |
| `PROJECT_GUIDELINES.docx` | Step-by-step project guide |

## Setup
```bash
pip install pandas numpy matplotlib seaborn jupyter
jupyter notebook capstone_notebook.ipynb
```

## Modules Covered
- Python Core & Intermediate (OOP, lambda, list comprehensions)
- NumPy (arrays, vectorised ops, statistics)
- Pandas (EDA, GroupBy, pivot tables, time series)
- Matplotlib (line, bar, pie, scatter, subplots)
- Seaborn (heatmap, boxplot, violin, pairplot, FacetGrid)
```


---
## ✅ Project Checklist

- [ ] Module 1 — Dataset loaded successfully  
- [ ] Module 2 — Data types and missing values identified  
- [ ] Module 3 — Data cleaned and derived columns added  
- [ ] Module 4 — Python OOP class implemented and tested  
- [ ] Module 5 — 5 NumPy operations completed  
- [ ] Module 6 — GroupBy, pivot table, RFM analysis done  
- [ ] Module 7 — 5 Matplotlib charts created and saved  
- [ ] Module 8 — 6 Seaborn charts created and saved  
- [ ] Module 9 — KPIs calculated and insights written  
- [ ] Module 10 — Code pushed to GitHub  

> 🎓 **Congratulations!** You've completed the Data Science Capstone Project.
